# AI Agent Security — E18 RowLock

Embeds the submitted `src/attack.py` and, on a competition rerun, serves the JED endpoint.
A normal commit run only writes `attack.py` and a placeholder `submission.csv`.


In [ ]:
import sys, os, glob, base64
from pathlib import Path

sys.argv = [sys.argv[0]]

for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    print(f'Dataset root: {dataset_root}')
    break

print('Setup complete')


In [ ]:
import base64
from pathlib import Path

ATTACK_PY_B64 = """..."""

attack_path = Path('/kaggle/working/attack.py')
attack_path.write_bytes(base64.b64decode(ATTACK_PY_B64))
print(f'Wrote {attack_path} ({attack_path.stat().st_size} bytes)')

src = attack_path.read_text(encoding='utf-8')
assert 'class AttackAlgorithm' in src
print('AttackAlgorithm present: OK')


In [ ]:
import os
from pathlib import Path

placeholder = (
    'Id,Score\n'
    'gpt_oss_public,0.0\n'
    'gpt_oss_private,0.0\n'
    'gemma_public,0.0\n'
    'gemma_private,0.0\n'
)
(Path('/kaggle/working') / 'submission.csv').write_text(placeholder)
print('submission.csv placeholder written')

# Host scoring sets KAGGLE_IS_COMPETITION_RERUN=1 and drives the gateway.
# A normal Save & Run All must not hang on serve().
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    from kaggle_evaluation.jed_attack_134815.jed_attack_inference_server import (
        JEDAttackInferenceServer,
    )
    print('Starting JEDAttackInferenceServer...')
    JEDAttackInferenceServer().serve()
else:
    print('Commit run: skip inference server (not a competition rerun).')
